In [18]:
import h5py
import numpy as np
import xarray as xr

# Process .mat file and convert to .nc 

file_path = "/glade/work/awells/air_quality/BMR/GBD_Country_Masks_0.10.mat"

with h5py.File(file_path, "r") as f:
    # Dereference mask objects
    mask_refs = f["sGBDCountries/Mask"][0]
    name_refs = f["sGBDCountries/Name"][0]
    region_refs = f["sGBDCountries/Region"][0]

    lat = f["gLAT"][0] 
    lon = f["gLON"][0]

    masks = []
    names = []
    regions = []

    for i, ref in enumerate(mask_refs):
        # Dereference mask
        mask = f[ref][()]
        masks.append(mask)

        # Country name
        name_ascii = f[name_refs[i]][()]
        name_str = "".join([chr(c[0]) for c in name_ascii])
        names.append(name_str)

        # Region
        region_ascii = f[region_refs[i]][()]
        region_str = "".join([chr(c[0]) for c in region_ascii])
        regions.append(region_str)

# Confirm all masks have same shape before stacking
first_shape = masks[0].shape
if all(mask.shape == first_shape for mask in masks):
    # Stack into 3D array: (country, lat, lon)
    mask_array = np.stack(masks)
    data_array = xr.DataArray(
        mask_array,
        dims=["country", "lat", "lon"],
        coords={
            "country": names,
            "lat": lat,
            "lon": lon,
            "region": ("country", regions),
        },
        name="country_mask"
    )
else:
    raise ValueError("Masks have varying shapes – cannot stack into DataArray.")

data_array.to_netcdf("/glade/work/awells/air_quality/BMR/GBD_Country_Masks_0.10.nc")

In [17]:
data_array


<xarray.DataArray 'country_mask' (country: 204, lat: 1800, lon: 3600)> Size: 1GB
array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
...
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]], dtype=int8)
Coordinates:
  * country  (country) <U32 26kB 'Armenia' 'Azerbaijan' ... 'Togo'
  * lat      (lat) float64 14kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon      (lon) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
    region   (country) <U28 23kB 'Central Asia' ... 'Western Sub-Saharan Africa'

In [14]:
len(lat)

1800

In [15]:
len(lon)

3600